In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import os


In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)
X = X / 255.0

# Use a 20k subset for SVM and KNN (they are slow on full 70k)
X_subset, _, y_subset, _ = train_test_split(
    X, y, train_size=20000, random_state=42, stratify=y
)

X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset, test_size=0.2, random_state=42, stratify=y_subset
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', C=1.0, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}


In [ ]:
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results[name] = {
        "accuracy": round(acc * 100, 2),
        "train_time": round(train_time, 2),
        "predictions": y_pred
    }
    print(f"  Accuracy: {acc*100:.2f}% | Time: {train_time:.2f}s")


In [ ]:
names = list(results.keys())
accuracies = [results[m]["accuracy"] for m in names]
times = [results[m]["train_time"] for m in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy bar chart
bars = axes[0].bar(names, accuracies, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title("Model Accuracy Comparison")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 105)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{acc}%", ha='center', fontweight='bold')

# Training time bar chart
bars2 = axes[1].bar(names, times, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[1].set_title("Training Time Comparison")
axes[1].set_ylabel("Time (seconds)")
for bar, t in zip(bars2, times):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{t}s", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("../data/model_comparison_chart.png")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result["predictions"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                ax=axes[idx],
                xticklabels=range(10),
                yticklabels=range(10))
    axes[idx].set_title(f"Confusion Matrix - {name}")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.suptitle("Confusion Matrices - All Models", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("../data/all_confusion_matrices.png")
plt.show()


In [ ]:
print("=" * 55)
print(f"{'Model':<20} {'Accuracy (%)':<18} {'Train Time (s)'}")
print("=" * 55)
for name, result in results.items():
    print(f"{name:<20} {result["accuracy"]:<18} {result["train_time"]}")
print("=" * 55)

best_model_name = max(results, key=lambda m: results[m]["accuracy"])
print(f"
Best Model: {best_model_name} "
      f"with {results[best_model_name]["accuracy"]}% accuracy")


In [ ]:
best_model = models[best_model_name]
os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/best_model.pkl")
print(f"Saved best model ({best_model_name}) to models/best_model.pkl")
